# MNIST To lakeFS

Download MNIST, engineer basic image features, write Parquet + metadata to lakeFS, sanity-read it back, and commit.

In [18]:
%pip install -q pandas numpy scikit-learn pyarrow s3fs requests

Note: you may need to restart the kernel to use updated packages.


In [19]:
import os

os.environ["LAKEFS_ENDPOINT"] = "http://3.142.147.224:8000"
os.environ["LAKEFS_ACCESS_KEY_ID"] = "local_access_key"
os.environ["LAKEFS_SECRET_ACCESS_KEY"] = "local_secret_key"
os.environ["LAKEFS_REPOSITORY"] = "qsentia-data"
os.environ["LAKEFS_BRANCH"] = "research"
os.environ["LAKEFS_SOURCE_BRANCH"] = "main"

In [20]:
import json
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import requests
import s3fs
from sklearn.datasets import fetch_openml

LAKEFS_ENDPOINT = os.environ["LAKEFS_ENDPOINT"].rstrip("/")
LAKEFS_ACCESS_KEY_ID = os.environ["LAKEFS_ACCESS_KEY_ID"]
LAKEFS_SECRET_ACCESS_KEY = os.environ["LAKEFS_SECRET_ACCESS_KEY"]
LAKEFS_REPOSITORY = os.environ["LAKEFS_REPOSITORY"]
LAKEFS_BRANCH = os.environ["LAKEFS_BRANCH"]
LAKEFS_SOURCE_BRANCH = os.environ["LAKEFS_SOURCE_BRANCH"]

RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S_utc")
DATASET_BASE = f"s3://{LAKEFS_REPOSITORY}/{LAKEFS_BRANCH}/datasets/mnist/source=openml/run_id={RUN_ID}"
FEATURE_BASE = f"s3://{LAKEFS_REPOSITORY}/{LAKEFS_BRANCH}/features/mnist/source=openml/run_id={RUN_ID}"

storage_options = {
    "key": LAKEFS_ACCESS_KEY_ID,
    "secret": LAKEFS_SECRET_ACCESS_KEY,
    "client_kwargs": {"endpoint_url": LAKEFS_ENDPOINT},
}

auth = (LAKEFS_ACCESS_KEY_ID, LAKEFS_SECRET_ACCESS_KEY)

print("repo:", LAKEFS_REPOSITORY)
print("branch:", LAKEFS_BRANCH)
print("run_id:", RUN_ID)

repo: qsentia-data
branch: research
run_id: 20260626_152159_utc


In [21]:
def ensure_branch(branch: str, source_branch: str) -> None:
    url = f"{LAKEFS_ENDPOINT}/api/v1/repositories/{LAKEFS_REPOSITORY}/branches/{branch}"
    response = requests.get(url, auth=auth, timeout=30)
    if response.status_code == 200:
        print(f"branch exists: {branch}")
        return

    create_url = f"{LAKEFS_ENDPOINT}/api/v1/repositories/{LAKEFS_REPOSITORY}/branches"
    create_response = requests.post(
        create_url,
        auth=auth,
        json={"name": branch, "source": source_branch},
        timeout=30,
    )
    create_response.raise_for_status()
    print(f"created branch: {branch} from {source_branch}")


ensure_branch(LAKEFS_BRANCH, LAKEFS_SOURCE_BRANCH)

branch exists: research


In [22]:
mnist = fetch_openml("mnist_784", version=1, as_frame=True, parser="auto")

pixels = mnist.data.astype("float32")
labels = mnist.target.astype("int64")

assert pixels.shape[1] == 784
assert len(pixels) == len(labels)

print("pixels:", pixels.shape)
print("labels:", sorted(labels.unique().tolist()))

pixels: (70000, 784)
labels: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]


In [23]:
def build_mnist_features(pixel_frame: pd.DataFrame, label_series: pd.Series) -> pd.DataFrame:
    pixel_values = pixel_frame.to_numpy(dtype=np.float32) / 255.0
    images = pixel_values.reshape(-1, 28, 28)

    row_grid = np.arange(28, dtype=np.float32).reshape(1, 28, 1)
    col_grid = np.arange(28, dtype=np.float32).reshape(1, 1, 28)
    pixel_sum = images.sum(axis=(1, 2))
    safe_pixel_sum = np.where(pixel_sum == 0, 1.0, pixel_sum)

    center_row = (images * row_grid).sum(axis=(1, 2)) / safe_pixel_sum
    center_col = (images * col_grid).sum(axis=(1, 2)) / safe_pixel_sum
    nonzero = images > 0
    rows_any = nonzero.any(axis=2)
    cols_any = nonzero.any(axis=1)

    top = np.argmax(rows_any, axis=1)
    bottom = 27 - np.argmax(rows_any[:, ::-1], axis=1)
    left = np.argmax(cols_any, axis=1)
    right = 27 - np.argmax(cols_any[:, ::-1], axis=1)
    empty = pixel_sum == 0

    return pd.DataFrame(
        {
            "label": label_series.to_numpy(dtype=np.int64),
            "pixel_mean": pixel_values.mean(axis=1),
            "pixel_std": pixel_values.std(axis=1),
            "pixel_sum": pixel_sum,
            "nonzero_pixels": nonzero.sum(axis=(1, 2)),
            "center_row": center_row,
            "center_col": center_col,
            "bbox_height": np.where(empty, 0, bottom - top + 1),
            "bbox_width": np.where(empty, 0, right - left + 1),
        }
    )


features = build_mnist_features(pixels, labels)

assert len(features) == len(pixels)
assert features.isna().sum().sum() == 0

features.head()

,label,pixel_mean,pixel_std,pixel_sum,nonzero_pixels,center_row,center_col,bbox_height,bbox_width
0,5,0.137680,0.312349,107.941193,166,14.035779,13.645189,20,20
1,0,0.155537,0.328970,121.941208,176,13.553298,14.097563,20,17
2,4,0.097254,0.257174,76.247055,120,14.103586,13.543175,20,20
3,1,0.085709,0.259133,67.196091,96,14.487538,13.993460,20,14
4,9,0.116116,0.291651,91.035286,142,14.079521,13.711811,20,15


In [24]:
raw_sample = pixels.head(1000).copy()
raw_sample.insert(0, "label", labels.head(1000).to_numpy(dtype=np.int64))

raw_path = f"{DATASET_BASE}/raw_sample.parquet"
features_path = f"{FEATURE_BASE}/features.parquet"
dataset_metadata_path = f"{DATASET_BASE}/_metadata.json"
feature_metadata_path = f"{FEATURE_BASE}/_metadata.json"

raw_sample.to_parquet(raw_path, index=False, storage_options=storage_options)
features.to_parquet(features_path, index=False, storage_options=storage_options)

fs = s3fs.S3FileSystem(**storage_options)
metadata = {
    "dataset": "mnist",
    "source": "openml",
    "run_id": RUN_ID,
    "created_at": datetime.now(timezone.utc).isoformat(),
    "format": "parquet",
    "raw_sample_rows": int(len(raw_sample)),
    "feature_rows": int(len(features)),
    "feature_columns": features.columns.tolist(),
}

for path in [dataset_metadata_path, feature_metadata_path]:
    with fs.open(path, "w") as file:
        json.dump(metadata, file, indent=2)

print("wrote", raw_path)
print("wrote", features_path)

wrote s3://qsentia-data/research/datasets/mnist/source=openml/run_id=20260626_152159_utc/raw_sample.parquet
wrote s3://qsentia-data/research/features/mnist/source=openml/run_id=20260626_152159_utc/features.parquet


In [25]:
readback = pd.read_parquet(features_path, storage_options=storage_options)

# Reading from partitioned lakeFS/S3 paths can add partition columns
# from path segments like source=openml and run_id=<RUN_ID>.
expected_columns = features.columns.tolist()

assert len(readback) == len(features)
assert set(expected_columns).issubset(set(readback.columns))
assert readback["label"].between(0, 9).all()

print("features:", features.shape)
print("readback:", readback.shape)
print("extra columns:", sorted(set(readback.columns) - set(expected_columns)))
readback.head()

features: (70000, 9)
readback: (70000, 11)
extra columns: ['run_id', 'source']


,label,pixel_mean,pixel_std,pixel_sum,nonzero_pixels,center_row,center_col,bbox_height,bbox_width,source,run_id
0,5,0.137680,0.312349,107.941193,166,14.035779,13.645189,20,20,openml,20260626_152159_utc
1,0,0.155537,0.328970,121.941208,176,13.553298,14.097563,20,17,openml,20260626_152159_utc
2,4,0.097254,0.257174,76.247055,120,14.103586,13.543175,20,20,openml,20260626_152159_utc
3,1,0.085709,0.259133,67.196091,96,14.487538,13.993460,20,14,openml,20260626_152159_utc
4,9,0.116116,0.291651,91.035286,142,14.079521,13.711811,20,15,openml,20260626_152159_utc


In [26]:
commit_url = f"{LAKEFS_ENDPOINT}/api/v1/repositories/{LAKEFS_REPOSITORY}/branches/{LAKEFS_BRANCH}/commits"
commit_payload = {
    "message": f"Add MNIST raw sample and engineered features {RUN_ID}",
    "metadata": {
        "dataset": "mnist",
        "source": "openml",
        "run_id": RUN_ID,
        "format": "parquet",
    },
}

response = requests.post(commit_url, auth=auth, json=commit_payload, timeout=30)
response.raise_for_status()

print(response.json())

{'committer': 'admin', 'creation_date': 1782487346, 'generation': 4, 'id': '547451427a14c29f4b302c3718e9e84bd685738a81ed52dab91ce5686250653b', 'message': 'Add MNIST raw sample and engineered features 20260626_152159_utc', 'meta_range_id': 'c97ca78e11920ede62e23ff0d8b98e47a97443e6cf13680da15f6834cfcebfdc', 'metadata': {'dataset': 'mnist', 'format': 'parquet', 'run_id': '20260626_152159_utc', 'source': 'openml'}, 'parents': ['1afe21d93833ce372e56548eb4a5a27e0163245b85df0290d0f70bc6b1c3d781'], 'version': 1}


Browse in lakeFS:

```text
datasets/mnist/source=openml/run_id=<RUN_ID>/
features/mnist/source=openml/run_id=<RUN_ID>/
```